In [ ]:
from pathlib import Path
from zipfile import ZipFile

import pandas as pd

from app.utils import unix_timestamp_to_datetime, stream_download

stream_download("https://www.okx.com/cdn/okex/traderecords/trades/daily/20250728/BTC-USDT-SWAP-trades-2025-07-28.zip", "raw_data/okx/swap/BTC-USDT-SWAP-trades-2025-07-28.zip")

def read_zip_csv(zip_path, csv_name=None):
    """
    Read a CSV file from a zip archive into a pandas DataFrame.
    
    Args:
        zip_path (str): Path to the zip file
        csv_name (str, optional): Name of CSV file inside zip. 
                                If None, assumes same name as zip but with .csv extension.
    
    Returns:
        pd.DataFrame: The loaded DataFrame
    """
    if csv_name is None:
        # Assume CSV has same name as zip but with .csv extension
        csv_name = zip_path.stem if hasattr(zip_path, 'stem') else Path(zip_path).stem
        csv_name += '.csv'
    
    with ZipFile(zip_path) as z:
        with z.open(csv_name) as f:
            return pd.read_csv(f, encoding="GBK")

file_path = Path("raw_data/okx/swap/BTC-USDT-SWAP-trades-2025-07-28.zip")
df = pd.read_csv(file_path, header=0, encoding="GBK", compression="zip")
df.columns = [col.split("/")[0] for col in df.columns]
df.sort_values(by=["created_time", "trade_id"], inplace=True)
df["created_time"] = df["created_time"].apply(unix_timestamp_to_datetime)
df.head(10)

,trade_id,side,size,price,created_time
1,1670367323,sell,0.06,118627.6,2025-07-27 16:00:00.118000+00:00
2,1670367324,sell,0.19,118627.6,2025-07-27 16:00:00.162000+00:00
891854,1670367325,sell,0.50,118627.6,2025-07-27 16:00:00.166000+00:00
891855,1670367326,sell,0.18,118627.6,2025-07-27 16:00:00.167000+00:00
891856,1670367327,sell,0.20,118627.6,2025-07-27 16:00:00.171000+00:00
3,1670367328,sell,0.20,118627.6,2025-07-27 16:00:00.190000+00:00
891857,1670367329,buy,0.01,118627.7,2025-07-27 16:00:00.228000+00:00
891858,1670367330,sell,1.76,118627.6,2025-07-27 16:00:00.433000+00:00
4,1670367331,sell,4.65,118627.6,2025-07-27 16:00:00.542000+00:00
5,1670367332,buy,1.01,118627.7,2025-07-27 16:00:00.557000+00:00


In [ ]:
from app.utils import stream_download, checksum

stream_download(
    "https://www.okx.com/cdn/okex/traderecords/trades/daily/20250728/BTC-USDT-trades-2025-07-28.zip",
    "raw_data/okx/spot/BTC-USDT-trades-2025-07-28.zip",
)
stream_download(
    "https://data.binance.vision/data/spot/daily/trades/BTCUSDT/BTCUSDT-trades-2025-07-28.zip",
    "raw_data/binance/spot/BTCUSDT-trades-2025-07-28.zip",
)
check, _ = checksum("raw_data/binance/spot/BTCUSDT-trades-2025-07-28.zip", "https://data.binance.vision/data/spot/daily/trades/BTCUSDT/BTCUSDT-trades-2025-07-28.zip.CHECKSUM")


(True,
 'checksum successful: raw_data/binance/spot/BTCUSDT-trades-2025-07-28.zip')

In [6]:
from app.utils import stream_download, checksum

stream_download(
    "https://data.binance.vision/data/spot/daily/klines/BTCUSDT/1m/BTCUSDT-1m-2025-07-28.zip",
    "raw_data/binance/spot/BTCUSDT-1m-2025-07-28.zip",
)
check, msg = checksum("raw_data/binance/spot/BTCUSDT-1m-2025-07-28.zip", "https://data.binance.vision/data/spot/daily/klines/BTCUSDT/1m/BTCUSDT-1m-2025-07-28.zip.CHECKSUM")
if not check:
   print(msg)

In [15]:
import pandas as pd

from app.utils import unix_timestamp_us_to_datetime

trades_df = pd.read_csv("raw_data/binance/spot/BTCUSDT-trades-2025-07-28.zip", header=None, compression="zip")
trades_df.columns = ["tradeId", "price", "quantity", "quoteQuantity", "time", "isBuyerMaker", "isBestMatch"]
trades_df["time"] = trades_df["time"].apply(unix_timestamp_us_to_datetime)

In [16]:
trades_df

,tradeId,price,quantity,quoteQuantity,time,isBuyerMaker,isBestMatch
0,5115339182,119415.56,0.00020,23.883112,2025-07-28 00:00:00.205797+00:00,False,True
1,5115339183,119415.55,0.05637,6731.454553,2025-07-28 00:00:00.218198+00:00,True,True
2,5115339184,119415.55,0.00664,792.919252,2025-07-28 00:00:00.351851+00:00,True,True
3,5115339185,119415.55,0.13770,16443.521235,2025-07-28 00:00:00.351851+00:00,True,True
4,5115339186,119415.55,0.14790,17661.559845,2025-07-28 00:00:00.351851+00:00,True,True
...,...,...,...,...,...,...,...
1824772,5117163954,118062.32,0.02673,3155.805814,2025-07-28 23:59:58.689019+00:00,False,True
1824773,5117163955,118062.32,0.00038,44.863682,2025-07-28 23:59:59.490137+00:00,False,True
1824774,5117163956,118062.32,0.00024,28.334957,2025-07-28 23:59:59.571970+00:00,False,True
1824775,5117163957,118062.32,0.00051,60.211783,2025-07-28 23:59:59.710054+00:00,False,True


In [ ]:
trades_df = trades_df.set_index("time")
# 创建主动买入标识和主动买入量列
trades_df['isTakerBuy'] = ~trades_df['isBuyerMaker']  # 反转布尔值
trades_df['takerBuyVolume'] = trades_df['quantity'] * trades_df['isTakerBuy']
trades_df['takerBuyAmount'] = trades_df['quoteQuantity'] * trades_df['isTakerBuy']
# 重采样并聚合计算分钟K线
kline_1m_df = trades_df.resample('min').agg({
    'price': ['first', 'max', 'min', 'last'],
    'quantity': 'sum',
    'quoteQuantity': 'sum',
    'tradeId': 'count',
    'takerBuyVolume': 'sum',
    'takerBuyAmount': 'sum', 
})

# 扁平化多级列索引
kline_1m_df.columns = ['_'.join(col).strip() for col in kline_1m_df.columns.values]

# 重命名列以匹配目标格式
kline_1m_df = kline_1m_df.rename(columns={
    'price_first': 'open',
    'price_max': 'high',
    'price_min': 'low',
    'price_last': 'close',
    'quantity_sum': 'volume',
    'quoteQuantity_sum': 'amount',
    'tradeId_count': 'trades',
    'takerBuyVolume_sum': 'takerBuyVolume',
    'takerBuyAmount_sum': 'takerBuyAmount'
})

# 添加ignore列并重置索引
kline_1m_df['ignore'] = 0
kline_1m_df = kline_1m_df.reset_index().rename(columns={'time': 'timestamp'})

# 选择并排序最终列
final_columns = ["timestamp", "open", "high", "low", "close", "volume", 
                 "amount", "trades", "takerBuyVolume", "takerBuyAmount", "ignore"]
kline_1m_df = kline_1m_df[final_columns]

# # 重采样计算主动买入量
# taker_buy = trades_df.resample("min").agg({
#     'quantity': lambda x: x[trades_df.loc[x.index, 'isTakerBuy']].sum(),
#     'quoteQuantity': lambda x: x[trades_df.loc[x.index, 'isTakerBuy']].sum()
# })

# # 合并结果
# kline_1m_df = kline_1m_df.join(taker_buy.rename(columns={
#     'quantity': 'takerBuyVolume',
#     'quoteQuantity': 'takerBuyAmount'
# }))

# # 添加ignore列并重置索引
# kline_1m_df['ignore'] = 0
# kline_1m_df = kline_1m_df.reset_index().rename(columns={'time': 'timestamp'})

# # 选择并排序最终列
# final_columns = ["timestamp", "open", "high", "low", "close", "volume", 
#                  "amount", "trades", "takerBuyVolume", "takerBuyAmount", "ignore"]
# kline_1m_df = [final_columns]

/var/folders/8j/r6qyy5kd06v9mx2vs7z30t880000gn/T/ipykernel_31001/520109097.py:7: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  kline_1m_df = trades_df.resample('1T').agg({


In [18]:
kline_1m_df

,timestamp,open,high,low,close,volume,amount,trades,takerBuyVolume,takerBuyAmount,ignore
0,2025-07-28 00:00:00+00:00,119415.56,119457.67,119365.48,119365.49,5.79730,692360.682289,1986,2.61698,312545.082947,0
1,2025-07-28 00:01:00+00:00,119365.48,119398.05,119365.48,119387.98,4.50031,537275.640918,1271,1.21154,144630.857254,0
2,2025-07-28 00:02:00+00:00,119387.99,119408.69,119387.59,119387.60,4.31437,515112.946411,1171,0.70594,84284.513765,0
3,2025-07-28 00:03:00+00:00,119387.60,119404.79,119385.15,119385.15,5.04521,602366.209153,886,2.25271,268959.225977,0
4,2025-07-28 00:04:00+00:00,119385.15,119385.16,119369.52,119369.52,2.77183,330883.456083,704,0.25753,30742.275582,0
...,...,...,...,...,...,...,...,...,...,...,...
1435,2025-07-28 23:55:00+00:00,117883.30,117883.30,117840.00,117840.01,5.48753,646763.113412,635,0.65493,77192.635770,0
1436,2025-07-28 23:56:00+00:00,117840.01,117861.97,117822.56,117835.47,3.08942,364055.762157,679,2.70698,318987.177959,0
1437,2025-07-28 23:57:00+00:00,117835.47,117905.13,117817.45,117905.12,4.38459,516697.772075,970,4.01823,473528.593065,0
1438,2025-07-28 23:58:00+00:00,117905.12,118000.11,117905.12,118000.10,3.74586,441818.584929,849,2.57138,303301.310426,0


In [28]:
df

,0,1,2,3,4,5,6,8,9,10,11
0,1753660800000000,119415.56,119457.67,119365.48,119365.49,5.79730,1753660859999999,692360.682289,1986,2.61698,312545.082947
1,1753660860000000,119365.48,119398.05,119365.48,119387.98,4.50031,1753660919999999,537275.640918,1271,1.21154,144630.857254
2,1753660920000000,119387.99,119408.69,119387.59,119387.60,4.31437,1753660979999999,515112.946411,1171,0.70594,84284.513765
3,1753660980000000,119387.60,119404.79,119385.15,119385.15,5.04521,1753661039999999,602366.209153,886,2.25271,268959.225977
4,1753661040000000,119385.15,119385.16,119369.52,119369.52,2.77183,1753661099999999,330883.456083,704,0.25753,30742.275582
...,...,...,...,...,...,...,...,...,...,...,...
1435,1753746900000000,117883.30,117883.30,117840.00,117840.01,5.48753,1753746959999999,646763.113412,635,0.65493,77192.635770
1436,1753746960000000,117840.01,117861.97,117822.56,117835.47,3.08942,1753747019999999,364055.762157,679,2.70698,318987.177959
1437,1753747020000000,117835.47,117905.13,117817.45,117905.12,4.38459,1753747079999999,516697.772075,970,4.01823,473528.593065
1438,1753747080000000,117905.12,118000.11,117905.12,118000.10,3.74586,1753747139999999,441818.584929,849,2.57138,303301.310425


In [ ]:
expected_kline_1m_df = pd.read_csv("raw_data/binance/spot/BTCUSDT-1m-2025-07-28.zip", header=None, compression="zip")
expected_kline_1m_df.columns = ["timestamp", "open", "high", "low", "close", "volume", "closeTime", "amount", "trades", "takerBuyVolume", "takerBuyAmount", "ignore"]
expected_kline_1m_df.drop(columns="closeTime", inplace=True)
expected_kline_1m_df["timestamp"] = expected_kline_1m_df["timestamp"].apply(unix_timestamp_us_to_datetime)
pd.testing.assert_frame_equal(expected_kline_1m_df, kline_1m_df)

,timestamp,open,high,low,close,volume,amount,trades,takerBuyVolume,takerBuyAmount,ignore
0,2025-07-28 00:00:00+00:00,119415.56,119457.67,119365.48,119365.49,5.79730,692360.682289,1986,2.61698,312545.082947,0
1,2025-07-28 00:01:00+00:00,119365.48,119398.05,119365.48,119387.98,4.50031,537275.640918,1271,1.21154,144630.857254,0
2,2025-07-28 00:02:00+00:00,119387.99,119408.69,119387.59,119387.60,4.31437,515112.946411,1171,0.70594,84284.513765,0
3,2025-07-28 00:03:00+00:00,119387.60,119404.79,119385.15,119385.15,5.04521,602366.209153,886,2.25271,268959.225977,0
4,2025-07-28 00:04:00+00:00,119385.15,119385.16,119369.52,119369.52,2.77183,330883.456083,704,0.25753,30742.275582,0
...,...,...,...,...,...,...,...,...,...,...,...
1435,2025-07-28 23:55:00+00:00,117883.30,117883.30,117840.00,117840.01,5.48753,646763.113412,635,0.65493,77192.635770,0
1436,2025-07-28 23:56:00+00:00,117840.01,117861.97,117822.56,117835.47,3.08942,364055.762157,679,2.70698,318987.177959,0
1437,2025-07-28 23:57:00+00:00,117835.47,117905.13,117817.45,117905.12,4.38459,516697.772075,970,4.01823,473528.593065,0
1438,2025-07-28 23:58:00+00:00,117905.12,118000.11,117905.12,118000.10,3.74586,441818.584929,849,2.57138,303301.310425,0


In [5]:
import pandas as pd

from app.utils import unix_timestamp_to_datetime

trades_df = pd.read_csv("raw_data/binance/spot/BTCUSDT-trades-2025-07-28.zip", header=None, compression="zip")
trades_df.columns = ["tradeId", "price", "quantity", "quoteQuantity", "time", "isBuyerMaker", "isBestMatch"]
trades_df["time"] = trades_df["time"].apply(lambda x: unix_timestamp_to_datetime(x, "us"))
trades_df = trades_df.set_index("time")
# 创建主动买入标识和主动买入量列
trades_df['isTakerBuy'] = ~trades_df['isBuyerMaker']  # 反转布尔值
trades_df['takerBuyVolume'] = trades_df['quantity'] * trades_df['isTakerBuy']
trades_df['takerBuyAmount'] = trades_df['quoteQuantity'] * trades_df['isTakerBuy']
# 重采样并聚合计算分钟K线
kline_1s_df = trades_df.resample('s').agg({
    'price': ['first', 'max', 'min', 'last'],
    'quantity': 'sum',
    'quoteQuantity': 'sum',
    'tradeId': 'count',
    'takerBuyVolume': 'sum',
    'takerBuyAmount': 'sum', 
})

# 扁平化多级列索引
kline_1s_df.columns = ['_'.join(col).strip() for col in kline_1s_df.columns.values]

# 重命名列以匹配目标格式
kline_1s_df = kline_1s_df.rename(columns={
    'price_first': 'open',
    'price_max': 'high',
    'price_min': 'low',
    'price_last': 'close',
    'quantity_sum': 'volume',
    'quoteQuantity_sum': 'amount',
    'tradeId_count': 'trades',
    'takerBuyVolume_sum': 'takerBuyVolume',
    'takerBuyAmount_sum': 'takerBuyAmount'
})

# 添加ignore列并重置索引
kline_1s_df['ignore'] = 0
kline_1s_df = kline_1s_df.reset_index().rename(columns={'time': 'timestamp'})

# 处理空值 - 用前一分钟的close填充当前分钟的价格
# 首先创建close列的ffill（前向填充）版本
kline_1s_df['close_ffill'] = kline_1s_df['close'].ffill()

# 标识需要填充的行（没有交易的行）
mask = kline_1s_df['open'].isna()

# 用前值close填充价格列
price_cols = ['open', 'high', 'low', 'close']
for col in price_cols:
    kline_1s_df.loc[mask, col] = kline_1s_df.loc[mask, 'close_ffill']

# 选择并排序最终列
final_columns = ["timestamp", "open", "high", "low", "close", "volume", 
                 "amount", "trades", "takerBuyVolume", "takerBuyAmount", "ignore"]
kline_1s_df = kline_1s_df[final_columns]


expected_kline_1s_df = pd.read_csv("raw_data/binance/spot/BTCUSDT-1s-2025-07-28.zip", header=None, compression="zip")
expected_kline_1s_df.columns = ["timestamp", "open", "high", "low", "close", "volume", "closeTime", "amount", "trades", "takerBuyVolume", "takerBuyAmount", "ignore"]
expected_kline_1s_df.drop(columns="closeTime", inplace=True)
expected_kline_1s_df["timestamp"] = expected_kline_1s_df["timestamp"].apply(lambda x: unix_timestamp_to_datetime(x, "us"))
pd.testing.assert_frame_equal(expected_kline_1s_df, kline_1s_df)

In [4]:
import pandas as pd

from app.utils import unix_timestamp_to_datetime

df_okx = pd.read_csv("raw_data/okx/spot/BTC-USDT-trades-2025-07-28.zip", header=0, encoding="GBK", compression="zip")
df_okx.columns = [col.split("/")[0] for col in df_okx.columns]
df_binance = pd.read_csv("raw_data/binance/spot/BTCUSDT-trades-2025-07-28.zip", header=None, compression="zip")
df_binance.columns = ["tradeId", "price", "quantity", "quoteQuantity", "time", "isBuyerMaker", "isBestMatch"]
unix_timestamp_to_datetime(df_okx["created_time"][0], unit="ms"), unix_timestamp_to_datetime(df_binance["time"][0], unit="us")

(datetime.datetime(2025, 7, 27, 16, 0, 0, 169000, tzinfo=<UTC>),
 datetime.datetime(2025, 7, 28, 0, 0, 0, 205797, tzinfo=<UTC>))